# EuroSAT EfficientNetB0 Kaggle Runner

This notebook runs Phase 5 EfficientNetB0 Stage 1 and Stage 2 on Kaggle.
It assumes dataset input is mounted under `/kaggle/input` and all outputs are written to `/kaggle/working`.

In [ ]:
import os
import shutil
from pathlib import Path

from kaggle_secrets import UserSecretsClient

WORKING = Path('/kaggle/working')
REPO = WORKING / 'satellite-land-classification-dl'

if REPO.exists():
    shutil.rmtree(REPO)

token = UserSecretsClient().get_secret('GH_TOKEN')
clone_url = f'https://{token}@github.com/milanovicmilos/satellite-land-classification-dl.git'

!git clone {clone_url} {REPO.as_posix()}
%cd {REPO.as_posix()}
!git checkout development

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -e .

## Dataset Path Setup

The default Kaggle input path in this notebook is `/kaggle/input/eurosat-dataset/EuroSAT`.
This folder must directly contain the EuroSAT class directories such as `AnnualCrop`, `Forest`, and `Residential`.

In [ ]:
import json

DATASET_INPUT_ROOT = Path('/kaggle/input/eurosat-dataset/EuroSAT')
STAGE1_RESUME_PATH = Path('/kaggle/working/checkpoints/efficientnet_b0/stage1/best_checkpoint.pt')
assert DATASET_INPUT_ROOT.exists(), f'Missing dataset path: {DATASET_INPUT_ROOT}'

CONFIG_STAGE1 = REPO / 'configs' / 'efficientnet_b0.stage1.json'
CONFIG_STAGE2 = REPO / 'configs' / 'efficientnet_b0.stage2.json'
DEFAULTS = REPO / 'configs' / 'experiment.defaults.json'

def patch_dataset_root(config_path: Path) -> None:
    payload = json.loads(config_path.read_text(encoding='utf-8'))
    payload['dataset_root'] = DATASET_INPUT_ROOT.as_posix()
    payload['training']['epochs'] = 30

    if 'stage2' in config_path.name:
        payload['training']['resume_from'] = STAGE1_RESUME_PATH.as_posix()

    config_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')

patch_dataset_root(CONFIG_STAGE1)
patch_dataset_root(CONFIG_STAGE2)
print('Patched config dataset_root to', DATASET_INPUT_ROOT)
print('Patched training epochs to 30 for full Kaggle runs')
print('Patched Stage 2 resume checkpoint to', STAGE1_RESUME_PATH)

## Prepare Deterministic Splits

In [ ]:
!python run.py --prepare-dataset --config configs/efficientnet_b0.stage1.json --defaults configs/experiment.defaults.json --splits-output /kaggle/working/artifacts/splits

## Stage 1 Smoke (Frozen Backbone)

In [ ]:
!python run.py --run-baseline --config configs/efficientnet_b0.stage1.json --defaults configs/experiment.defaults.json --splits-output /kaggle/working/artifacts/splits --reports-output /kaggle/working/artifacts/reports/efficientnet_b0_stage1_smoke.json --checkpoints-output /kaggle/working/checkpoints/efficientnet_b0/stage1

## Stage 2 Smoke (Unfrozen Backbone, Resume from Stage 1)

The config patch cell rewrites `resume_from` to the absolute Kaggle path `/kaggle/working/checkpoints/efficientnet_b0/stage1/best_checkpoint.pt`.
Run Stage 1 first so the Stage 2 resume checkpoint exists before starting the unfrozen pass.

In [ ]:
!python run.py --run-baseline --config configs/efficientnet_b0.stage2.json --defaults configs/experiment.defaults.json --splits-output /kaggle/working/artifacts/splits --reports-output /kaggle/working/artifacts/reports/efficientnet_b0_stage2_smoke.json --checkpoints-output /kaggle/working/checkpoints/efficientnet_b0/stage2

## Optional Full Runs

After smoke validation, the patched configs now run 30 epochs. Re-run Stage 1 first, then Stage 2.

In [ ]:
!zip -r /kaggle/working/results.zip /kaggle/working/artifacts /kaggle/working/checkpoints